In [1]:
import os
import json
import re
import sys
from pathlib import Path
from datetime import date, datetime, timedelta
from dotenv import load_dotenv

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_groq import ChatGroq

load_dotenv()

# Add backend to path so we can reuse legal_graph utilities
sys.path.append("../backend")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("Imports OK")
print(f"Today's date: {date.today()}")

python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 19


Imports OK
Today's date: 2026-06-06


In [2]:
from supabase import create_client, Client

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_ANON_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_ANON_KEY must be set in your .env file")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase connected")

# ── Create the contracts table (run once) ─────────────────────────────────────
# Go to your Supabase dashboard → SQL Editor and run this SQL:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS contracts (
    id                UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    user_id           TEXT NOT NULL DEFAULT 'default',
    filename          TEXT NOT NULL,
    party_a           TEXT,
    party_b           TEXT,
    contract_type     TEXT,
    start_date        DATE,
    end_date          DATE,
    renewal_date      DATE,
    notice_period_days INT DEFAULT 30,
    auto_renewal      BOOLEAN DEFAULT FALSE,
    governing_law     TEXT DEFAULT 'Indian Law',
    key_obligations   TEXT,
    raw_text          TEXT,
    reminder_sent_30  BOOLEAN DEFAULT FALSE,
    reminder_sent_7   BOOLEAN DEFAULT FALSE,
    reminder_sent_1   BOOLEAN DEFAULT FALSE,
    created_at        TIMESTAMPTZ DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_contracts_renewal ON contracts(renewal_date);
CREATE INDEX IF NOT EXISTS idx_contracts_user    ON contracts(user_id);
"""

print("Copy the SQL above and run it in your Supabase dashboard → SQL Editor")
print("(Only needed once — skip if table already exists)")

Supabase connected
Copy the SQL above and run it in your Supabase dashboard → SQL Editor
(Only needed once — skip if table already exists)


In [3]:
def extract_text_from_pdf(file_path: str, max_pages: int = 8) -> str:
    """
    Extract and clean text from a contract PDF.
    Uses first max_pages pages — key dates are almost always in the first 8 pages.
    """
    loader = PyMuPDFLoader(file_path)
    pages  = loader.load()

    text_parts = []
    for page in pages[:max_pages]:
        # Remove non-ASCII noise (Hindi/Unicode artifacts)
        text = "".join(ch for ch in page.page_content if ord(ch) < 128)
        # Remove footer separators
        text = re.split(r"_{2,}", text)[0].strip()
        # Collapse excessive whitespace
        text = re.sub(r"\n{3,}", "\n\n", text)
        if text:
            text_parts.append(text)

    full_text = "\n\n".join(text_parts)
    print(f"Extracted {len(pages)} pages, using first {min(len(pages), max_pages)}")
    print(f"Total characters extracted: {len(full_text)}")
    return full_text


# ── Test with a sample contract ───────────────────────────────────────────────
# CHANGE THIS PATH to any contract PDF you have
SAMPLE_CONTRACT_PATH = "../data/raw/sample_contract.pdf"

if Path(SAMPLE_CONTRACT_PATH).exists():
    sample_text = extract_text_from_pdf(SAMPLE_CONTRACT_PATH)
    print("\n── First 500 characters ──")
    print(sample_text[:500])
else:
    print(f"File not found: {SAMPLE_CONTRACT_PATH}")
    print("   Place any contract PDF at that path and rerun, or update the path above")
    # Use a dummy text for the rest of the notebook to still demonstrate the flow
    sample_text = """
    SERVICE AGREEMENT

    This Service Agreement ("Agreement") is entered into as of January 1, 2025,
    between Acme Technologies Pvt. Ltd. ("Party A") and BuildRight Solutions Pvt. Ltd. ("Party B").

    Term: This Agreement shall commence on January 1, 2025 and shall continue
    until December 31, 2025, unless earlier terminated.

    Renewal: This Agreement shall automatically renew for successive one-year periods
    unless either party provides written notice of non-renewal at least 30 days
    prior to the end of the then-current term.

    Governing Law: This Agreement shall be governed by the laws of India,
    and any disputes shall be subject to the exclusive jurisdiction of the
    courts in Bangalore, Karnataka.

    Key Obligations: Party A shall provide software development services.
    Party B shall make monthly payments of INR 5,00,000 by the 5th of each month.
    """
    print("   Using dummy contract text for demonstration")

File not found: ../data/raw/sample_contract.pdf
   Place any contract PDF at that path and rerun, or update the path above
   Using dummy contract text for demonstration


In [4]:
EXTRACT_PROMPT = """You are a legal contract analyst specializing in Indian commercial contracts.
Extract the following fields from the contract text provided.

Return ONLY a valid JSON object with these exact keys — no markdown, no explanation, no preamble:

{{
  "party_a": "full legal name of the first party",
  "party_b": "full legal name of the second party",
  "contract_type": "one of: Service Agreement, Employment Agreement, Lease Agreement, NDA, Loan Agreement, Sale Agreement, Partnership Agreement, Other",
  "start_date": "YYYY-MM-DD or null if not found",
  "end_date": "YYYY-MM-DD or null if not found",
  "renewal_date": "YYYY-MM-DD — same as end_date if auto-renewal, else null",
  "notice_period_days": <integer number of days, default 30 if not mentioned>,
  "auto_renewal": <true if contract mentions automatic renewal, false otherwise>,
  "governing_law": "jurisdiction mentioned, e.g. Laws of India / Maharashtra / Delhi",
  "key_obligations": "2-3 sentence summary of main obligations of each party"
}}

Important rules:
- Dates MUST be in YYYY-MM-DD format. Convert "31st March 2026" to "2026-03-31".
- If a date is ambiguous or missing, use null — do not guess.
- If auto_renewal is true, set renewal_date = end_date.
- Return ONLY the JSON. No ```json fences. No extra text.

Contract text:
{text}"""


def extract_contract_data(file_path: str) -> dict:
    """
    Full pipeline: PDF → cleaned text → LLM extraction → validated dict.
    """
    # Step 1: Extract text
    text = extract_text_from_pdf(file_path)

    # Step 2: Truncate to 8000 chars to stay within LLM context limits
    # Key contract dates are always in the first portion
    truncated = text[:8000]

    # Step 3: Call LLM
    prompt   = EXTRACT_PROMPT.format(text=truncated)
    response = llm.invoke(prompt).content.strip()

    # Step 4: Clean LLM response — strip markdown fences if present
    response = re.sub(r"```(?:json)?|```", "", response).strip()

    # Step 5: Parse JSON
    try:
        data = json.loads(response)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw LLM response was:\n{response}")
        raise

    # Step 6: Validate and normalise dates
    for date_field in ["start_date", "end_date", "renewal_date"]:
        val = data.get(date_field)
        if val:
            try:
                datetime.strptime(val, "%Y-%m-%d")  # validate format
            except ValueError:
                print(f"Invalid date format for {date_field}: '{val}' — setting to null")
                data[date_field] = None

    # Step 7: Set renewal_date = end_date when auto_renewal is True and renewal_date missing
    if data.get("auto_renewal") and not data.get("renewal_date"):
        data["renewal_date"] = data.get("end_date")

    return data


print("Extraction function defined")

Extraction function defined


In [5]:
# ── Run extraction on the sample text directly (no file needed) ───────────────
def extract_from_text(text: str) -> dict:
    """Same as extract_contract_data but accepts raw text (for testing)."""
    truncated = text[:8000]
    prompt    = EXTRACT_PROMPT.format(text=truncated)
    response  = llm.invoke(prompt).content.strip()
    response  = re.sub(r"```(?:json)?|```", "", response).strip()

    try:
        data = json.loads(response)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw response:\n{response}")
        raise

    for date_field in ["start_date", "end_date", "renewal_date"]:
        val = data.get(date_field)
        if val:
            try:
                datetime.strptime(val, "%Y-%m-%d")
            except ValueError:
                print(f"⚠️  Bad date {date_field}: {val} — nulling")
                data[date_field] = None

    if data.get("auto_renewal") and not data.get("renewal_date"):
        data["renewal_date"] = data.get("end_date")

    return data


extracted = extract_from_text(sample_text)

print("Extraction result:")
print(json.dumps(extracted, indent=2))

Extraction result:
{
  "party_a": "Acme Technologies Pvt. Ltd.",
  "party_b": "BuildRight Solutions Pvt. Ltd.",
  "contract_type": "Service Agreement",
  "start_date": "2025-01-01",
  "end_date": "2025-12-31",
  "renewal_date": "2025-12-31",
  "notice_period_days": 30,
  "auto_renewal": true,
  "governing_law": "Laws of India",
  "key_obligations": "Party A shall provide software development services and Party B shall make monthly payments of INR 5,00,000 by the 5th of each month, thus outlining the primary responsibilities of both parties in this agreement."
}


In [7]:
def save_contract_to_supabase(
    data: dict,
    filename: str,
    user_id: str = "default",
    raw_text: str = "",
) -> dict:
    """
    Insert extracted contract data into Supabase contracts table.
    Returns the inserted row including the generated UUID.
    """
    record = {
        "user_id":            user_id,
        "filename":           filename,
        "party_a":            data.get("party_a"),
        "party_b":            data.get("party_b"),
        "contract_type":      data.get("contract_type"),
        "start_date":         data.get("start_date"),
        "end_date":           data.get("end_date"),
        "renewal_date":       data.get("renewal_date"),
        "notice_period_days": data.get("notice_period_days", 30),
        "auto_renewal":       data.get("auto_renewal", False),
        "governing_law":      data.get("governing_law", "Indian Law"),
        "key_obligations":    data.get("key_obligations"),
        "raw_text":           raw_text[:5000],  # cap to avoid Supabase row size limits
    }

    # Remove None values — Supabase inserts NULL for missing keys anyway
    record = {k: v for k, v in record.items() if v is not None}

    response = supabase.table("contracts").insert(record).execute()

    if not response.data:
        raise RuntimeError(f"Supabase insert failed: {response}")

    inserted = response.data[0]
    print(f"Saved to Supabase")
    print(f"Contract ID : {inserted['id']}")
    print(f"Parties : {inserted.get('party_a')} ↔ {inserted.get('party_b')}")
    print(f"Renewal date: {inserted.get('renewal_date')}")
    return inserted


# ── Save the extracted sample to Supabase ────────────────────────────────────
saved_record = save_contract_to_supabase(
    data=extracted,
    filename="sample_contract.pdf",
    user_id="notebook_test",
    raw_text=sample_text,
)

Saved to Supabase
Contract ID : dfabc711-3ec1-4870-a2cd-53cf973748e1
Parties : Acme Technologies Pvt. Ltd. ↔ BuildRight Solutions Pvt. Ltd.
Renewal date: 2025-12-31


In [8]:
def check_upcoming_renewals(
    user_id: str = "default",
    days_ahead_list: list = [30, 7, 1],
) -> list:
    """
    Check which contracts are renewing in the next N days
    and haven't had reminders sent yet.
    Returns a list of (contract, days_ahead) tuples that need reminders.
    """
    today   = date.today()
    pending = []

    for days_ahead in days_ahead_list:
        target     = today + timedelta(days=days_ahead)
        col_name   = f"reminder_sent_{days_ahead}"

        response = (
            supabase.table("contracts")
            .select("*")
            .eq("user_id", user_id)
            .eq("renewal_date", str(target))
            .eq(col_name, False)
            .execute()
        )

        for contract in response.data:
            pending.append((contract, days_ahead))
            print(f"⚠️  Reminder due: '{contract['contract_type']}' renews in {days_ahead} day(s)")
            print(f"    Parties : {contract.get('party_a')} ↔ {contract.get('party_b')}")
            print(f"    Renewal : {contract.get('renewal_date')}")

    if not pending:
        print(f"✅ No reminders due today ({today})")

    return pending


# ── Test: check renewals for our test user ────────────────────────────────────
pending_reminders = check_upcoming_renewals(user_id="notebook_test")
print(f"\nTotal pending reminders: {len(pending_reminders)}")

✅ No reminders due today (2026-06-06)

Total pending reminders: 0


In [9]:
# Cell 8 — Notification Delivery (Gmail SMTP — no paid services needed)

import smtplib
import threading
import time
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# ── Pull credentials from .env ────────────────────────────────────────────────
EMAIL_SENDER   = os.getenv("EMAIL_SENDER")    # your Gmail address
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # Gmail App Password (not your login password)
EMAIL_RECEIVER = os.getenv("EMAIL_RECEIVER")  # where reminders are sent


def send_free_reminder_email(
    contract: dict,
    days_ahead,          # int (30/7/1) for scheduled reminders, or str ("TEST") for test emails
    to_email: str,
) -> bool:
    """
    Send a contract renewal reminder email via Gmail SMTP.
    Uses Python's built-in smtplib — no paid email service required.

    Args:
        contract:   Contract dict from Supabase (or extracted data dict)
        days_ahead: Number of days until renewal, or "TEST" for instant test emails
        to_email:   Recipient email address

    Returns:
        True if email sent successfully, False otherwise
    """
    if not EMAIL_SENDER or not EMAIL_PASSWORD:
        print("❌ EMAIL_SENDER or EMAIL_PASSWORD not set in .env — skipping email")
        return False

    # ── Build subject line ────────────────────────────────────────────────────
    if str(days_ahead).upper() == "TEST":
        subject = f"[TEST] LexRAG Contract Intelligence — Pipeline Verification Email"
    else:
        subject = (
            f"⚠️ Contract Renewal in {days_ahead} Day{'s' if int(days_ahead) > 1 else ''}: "
            f"{contract.get('contract_type', 'Agreement')}"
        )

    # ── Renewal date display ──────────────────────────────────────────────────
    renewal_display = contract.get("renewal_date") or "Not specified"
    auto_renewal    = "Yes ✅" if contract.get("auto_renewal") else "No ❌"

    # ── Days badge colour — red for urgent, amber for 7-day, green for 30-day ─
    if str(days_ahead).upper() == "TEST":
        badge_color = "#2E75B6"
        badge_label = "TEST EMAIL"
    elif int(days_ahead) <= 1:
        badge_color = "#C0392B"
        badge_label = f"{days_ahead} DAY LEFT"
    elif int(days_ahead) <= 7:
        badge_color = "#E67E22"
        badge_label = f"{days_ahead} DAYS LEFT"
    else:
        badge_color = "#1A5276"
        badge_label = f"{days_ahead} DAYS LEFT"

    # ── HTML body ─────────────────────────────────────────────────────────────
    html_body = f"""
    <html>
    <body style="margin:0; padding:0; background-color:#F4F6F9; font-family: Arial, sans-serif;">

      <div style="max-width:620px; margin:30px auto; background:#FFFFFF;
                  border-radius:8px; overflow:hidden;
                  box-shadow: 0 2px 8px rgba(0,0,0,0.1);">

        <!-- Header -->
        <div style="background-color:#1F3864; padding:24px 32px;">
          <h1 style="margin:0; color:#FFFFFF; font-size:22px; font-weight:bold;">
            ⚖️ LexRAG Contract Intelligence
          </h1>
          <p style="margin:6px 0 0; color:#A9CCE3; font-size:13px;">
            Automated Contract Renewal Notification
          </p>
        </div>

        <!-- Badge -->
        <div style="background-color:{badge_color}; padding:12px 32px; text-align:center;">
          <span style="color:#FFFFFF; font-size:16px; font-weight:bold; letter-spacing:2px;">
            {badge_label}
          </span>
        </div>

        <!-- Body -->
        <div style="padding:28px 32px;">
          <p style="color:#333333; font-size:15px; margin-top:0;">
            {"This is an automated test email confirming your LexRAG contract pipeline is working correctly." 
              if str(days_ahead).upper() == "TEST" 
              else f"The following contract requires your attention. It is due for renewal in <strong>{days_ahead} day{'s' if int(days_ahead) > 1 else ''}</strong>."}
          </p>

          <!-- Contract Details Table -->
          <table style="width:100%; border-collapse:collapse; margin-top:16px; font-size:14px;">

            <tr style="background-color:#EAF2FB;">
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864; width:38%; border-bottom:1px solid #D5E8F5;">
                Contract Type
              </td>
              <td style="padding:10px 14px; color:#333333; border-bottom:1px solid #D5E8F5;">
                {contract.get("contract_type", "N/A")}
              </td>
            </tr>

            <tr>
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864; border-bottom:1px solid #EEEEEE;">
                Party A
              </td>
              <td style="padding:10px 14px; color:#333333; border-bottom:1px solid #EEEEEE;">
                {contract.get("party_a", "N/A")}
              </td>
            </tr>

            <tr style="background-color:#EAF2FB;">
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864; border-bottom:1px solid #D5E8F5;">
                Party B
              </td>
              <td style="padding:10px 14px; color:#333333; border-bottom:1px solid #D5E8F5;">
                {contract.get("party_b", "N/A")}
              </td>
            </tr>

            <tr>
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864; border-bottom:1px solid #EEEEEE;">
                Renewal Date
              </td>
              <td style="padding:10px 14px; color:{badge_color}; font-weight:bold; border-bottom:1px solid #EEEEEE;">
                {renewal_display}
              </td>
            </tr>

            <tr style="background-color:#EAF2FB;">
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864; border-bottom:1px solid #D5E8F5;">
                Auto Renewal
              </td>
              <td style="padding:10px 14px; color:#333333; border-bottom:1px solid #D5E8F5;">
                {auto_renewal}
              </td>
            </tr>

            <tr>
              <td style="padding:10px 14px; font-weight:bold; color:#1F3864;">
                Key Obligations
              </td>
              <td style="padding:10px 14px; color:#555555; line-height:1.6;">
                {contract.get("key_obligations", "Not specified")}
              </td>
            </tr>

          </table>

          <!-- Action note -->
          <div style="margin-top:24px; padding:14px 16px; background:#FFF3CD;
                      border-left:4px solid #E67E22; border-radius:4px;">
            <p style="margin:0; color:#7D6608; font-size:13px;">
              <strong>Action Required:</strong>
              {"This is a test email — no action needed." 
                if str(days_ahead).upper() == "TEST" 
                else f"Please review and decide whether to renew, renegotiate, or terminate this contract before the renewal date."}
            </p>
          </div>
        </div>

        <!-- Footer -->
        <div style="background-color:#F4F6F9; padding:16px 32px; text-align:center;
                    border-top:1px solid #DDDDDD;">
          <p style="margin:0; color:#999999; font-size:12px;">
            Generated by LexRAG Contract Intelligence &nbsp;|&nbsp;
            Sent: {datetime.now().strftime("%d %b %Y at %H:%M:%S")}
          </p>
        </div>

      </div>
    </body>
    </html>
    """

    # ── Build MIME message ────────────────────────────────────────────────────
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"]    = f"LexRAG Contracts <{EMAIL_SENDER}>"
    msg["To"]      = to_email
    msg.attach(MIMEText(html_body, "html"))

    # ── Send via Gmail SMTP ───────────────────────────────────────────────────
    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.ehlo()                            # identify ourselves to the server
            server.starttls()                        # upgrade connection to TLS encryption
            server.ehlo()                            # re-identify after TLS upgrade
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_SENDER, to_email, msg.as_string())

        print(f"✅ Email sent → {to_email}  (days_ahead={days_ahead})")
        return True

    except smtplib.SMTPAuthenticationError:
        print("❌ SMTP Authentication failed.")
        print("   Make sure EMAIL_PASSWORD is a Gmail App Password, not your Gmail login password.")
        print("   Generate one at: https://myaccount.google.com/apppasswords")
        return False

    except smtplib.SMTPException as e:
        print(f"❌ SMTP error: {e}")
        return False

    except Exception as e:
        print(f"❌ Unexpected error sending email: {e}")
        return False


# ── Background thread worker ──────────────────────────────────────────────────
def delayed_test_email_worker(
    contract: dict,
    to_email: str,
    delay_seconds: int = 120,
) -> None:
    """
    Runs inside a background daemon thread.
    Sleeps for delay_seconds, then fires a test email.

    This runs completely independently of the main notebook thread —
    the cell that spawns this finishes instantly and you can keep
    working while this waits in the background.
    """
    thread_name = threading.current_thread().name
    print(f"\n[{thread_name}] 🕐 Background thread started")
    print(f"[{thread_name}] Will fire test email in {delay_seconds} seconds ({delay_seconds//60} min)...")

    # Non-blocking sleep — only this thread pauses, not the notebook
    time.sleep(delay_seconds)

    print(f"\n[{thread_name}] ⏰ Delay complete — firing test email now...")
    success = send_free_reminder_email(
        contract=contract,
        days_ahead="TEST",
        to_email=to_email,
    )

    if success:
        print(f"[{thread_name}] ✅ Test email delivered to {to_email}")
    else:
        print(f"[{thread_name}] ❌ Test email failed — check EMAIL_SENDER / EMAIL_PASSWORD in .env")

    print(f"[{thread_name}] 🏁 Background thread finished")


# ── Console-only fallback (no email setup needed for testing) ─────────────────
def print_reminder(contract: dict, days_ahead) -> None:
    """Fallback: print reminder to notebook output instead of sending email."""
    print("\n" + "=" * 60)
    print(f"  CONTRACT RENEWAL REMINDER — {days_ahead} DAY(S) REMAINING")
    print("=" * 60)
    print(f"  Type         : {contract.get('contract_type', 'N/A')}")
    print(f"  Party A      : {contract.get('party_a', 'N/A')}")
    print(f"  Party B      : {contract.get('party_b', 'N/A')}")
    print(f"  Renewal Date : {contract.get('renewal_date', 'N/A')}")
    print(f"  Auto Renewal : {'Yes' if contract.get('auto_renewal') else 'No'}")
    print(f"  Obligations  : {contract.get('key_obligations', 'N/A')}")
    print("=" * 60 + "\n")


print("✅ Cell 8 loaded — Gmail SMTP email functions ready")
print(f"   EMAIL_SENDER   : {EMAIL_SENDER or '⚠️  not set in .env'}")
print(f"   EMAIL_RECEIVER : {EMAIL_RECEIVER or '⚠️  not set in .env'}")
print(f"   EMAIL_PASSWORD : {'set ✅' if EMAIL_PASSWORD else '⚠️  not set in .env'}")

✅ Cell 8 loaded — Gmail SMTP email functions ready
   EMAIL_SENDER   : mittalishank@gmail.com
   EMAIL_RECEIVER : mittalishank@gmail.com
   EMAIL_PASSWORD : set ✅


In [10]:
def mark_reminder_sent(contract_id: str, days_ahead: int) -> None:
    """
    After sending a reminder, mark it as sent in Supabase
    so it doesn't fire again tomorrow.
    """
    col_name = f"reminder_sent_{days_ahead}"
    supabase.table("contracts").update({col_name: True}).eq("id", contract_id).execute()
    print(f"✅ Marked {col_name}=True for contract {contract_id[:8]}...")


def run_reminder_job(user_id: str = "default", to_email: str = None) -> None:
    """
    Full reminder job:
    1. Find contracts needing reminders
    2. Send email (or print) for each
    3. Mark as sent in Supabase
    
    This is called by APScheduler daily at 8 AM in the FastAPI backend.
    """
    pending = check_upcoming_renewals(user_id=user_id)

    if not pending:
        return

    for contract, days_ahead in pending:
        # Print reminder always (useful for logging)
        print_reminder(contract, days_ahead)

        # Send email if configured
        if to_email and os.getenv("SENDGRID_API_KEY"):
            sent = send_reminder_email_sendgrid(contract, days_ahead, to_email)
        else:
            print("No email configured — printed reminder above")
            sent = True  # count print as "sent" for marking purposes

        if sent:
            mark_reminder_sent(contract["id"], days_ahead)


# ── Dry run (prints reminders without marking as sent) ────────────────────────
print("Running dry-run reminder check...")
pending = check_upcoming_renewals(user_id="notebook_test")
for contract, days_ahead in pending:
    print_reminder(contract, days_ahead)

Running dry-run reminder check...
✅ No reminders due today (2026-06-06)


In [11]:
def query_contracts(question: str, user_id: str = "default") -> str:
    """
    Answer natural language questions about stored contracts.
    Fetches all contracts for the user, formats them as context,
    then asks the LLM to answer based only on that context.
    """
    # Fetch all contracts for the user
    response = (
        supabase.table("contracts")
        .select(
            "id, filename, party_a, party_b, contract_type, "
            "start_date, end_date, renewal_date, notice_period_days, "
            "auto_renewal, governing_law, key_obligations"
        )
        .eq("user_id", user_id)
        .order("created_at", desc=True)
        .execute()
    )

    contracts = response.data

    if not contracts:
        return f"No contracts found for user '{user_id}'."

    # Format contracts as readable context
    context_parts = []
    for i, c in enumerate(contracts, 1):
        context_parts.append(
            f"Contract {i} ({c.get('contract_type', 'Unknown')}):\n"
            f"  File: {c.get('filename')}\n"
            f"  Parties: {c.get('party_a')} and {c.get('party_b')}\n"
            f"  Start: {c.get('start_date')} | End: {c.get('end_date')}\n"
            f"  Renewal: {c.get('renewal_date')} | Auto-renewal: {c.get('auto_renewal')}\n"
            f"  Notice period: {c.get('notice_period_days')} days\n"
            f"  Governing law: {c.get('governing_law')}\n"
            f"  Key obligations: {c.get('key_obligations')}"
        )

    context = "\n\n".join(context_parts)

    prompt = (
        "You are a contract management assistant.\n"
        "Answer the question using ONLY the contract data provided below.\n"
        "Be specific — mention party names, dates, and contract types.\n"
        "If the answer is not in the data, say so clearly.\n\n"
        f"Contract data:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    answer = llm.invoke(prompt).content
    return answer


# ── Test natural language queries ─────────────────────────────────────────────
test_questions = [
    "Which contracts are expiring soon?",
    "What are my obligations under the service agreement?",
    "Which contracts have auto-renewal clauses?",
    "What is the governing law for my contracts?",
]

print("Testing contract queries...\n")
for q in test_questions:
    print(f"Q: {q}")
    answer = query_contracts(q, user_id="notebook_test")
    print(f"A: {answer}\n")
    print("-" * 60)

Testing contract queries...

Q: Which contracts are expiring soon?
A: Based on the provided contract data, Contract 1 (Service Agreement) between Acme Technologies Pvt. Ltd. and BuildRight Solutions Pvt. Ltd. is expiring on 2025-12-31. However, since Auto-renewal is set to True, it will automatically renew on the same date unless a notice of termination is given within the 30-day notice period.

------------------------------------------------------------
Q: What are my obligations under the service agreement?
A: According to Contract 1 (Service Agreement) between Acme Technologies Pvt. Ltd. and BuildRight Solutions Pvt. Ltd., the key obligations are as follows: 

If you are Acme Technologies Pvt. Ltd. (Party A), your obligation is to provide software development services. 

If you are BuildRight Solutions Pvt. Ltd. (Party B), your obligation is to make monthly payments of INR 5,00,000 by the 5th of each month. 

However, without knowing which party you represent, I can only provide th

In [12]:
# Cell 11 — End-to-End Pipeline (with 2-minute background test email)

def full_contract_pipeline(
    file_path: str,
    user_id: str = "default",
) -> dict:
    """
    Complete contract intelligence pipeline:

    Step 1 → Extract raw text from the PDF
    Step 2 → LLM extracts structured metadata (parties, dates, obligations)
    Step 3 → Save structured record to Supabase
    Step 4 → Spawn background thread: fires a test email after 2 minutes
    Step 5 → Check if any scheduled reminders are due today
    Step 6 → Return immediately — no waiting for the background thread

    Args:
        file_path: Path to the contract PDF
        user_id:   Identifier for the user (used to scope Supabase queries)

    Returns:
        The Supabase record dict for the saved contract
    """
    print(f"\n{'=' * 60}")
    print(f"  CONTRACT INTELLIGENCE PIPELINE")
    print(f"  File    : {Path(file_path).name}")
    print(f"  User    : {user_id}")
    print(f"  Started : {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'=' * 60}")

    # ── Step 1: Extract raw text from PDF ────────────────────────────────────
    print("\n[1/5] Extracting text from PDF...")
    raw_text = extract_text_from_pdf(file_path)

    # ── Step 2: LLM extracts structured metadata ──────────────────────────────
    print("\n[2/5] Running LLM extraction...")
    data = extract_from_text(raw_text)
    print("      Extracted fields:")
    for key, val in data.items():
        if key != "key_obligations":          # obligations can be long — print separately
            print(f"        {key:<22} : {val}")
    print(f"        {'key_obligations':<22} : {str(data.get('key_obligations', ''))[:80]}...")

    # ── Step 3: Save to Supabase ──────────────────────────────────────────────
    print("\n[3/5] Saving to Supabase...")
    record = save_contract_to_supabase(
        data=data,
        filename=Path(file_path).name,
        user_id=user_id,
        raw_text=raw_text,
    )
    print(f"      Contract ID : {record['id']}")
    print(f"      Renewal Date: {record.get('renewal_date', 'not found')}")

    # ── Step 4: Spawn background test email thread ────────────────────────────
    print("\n[4/5] Scheduling background test email...")
    target_email = os.getenv("EMAIL_RECEIVER")

    if target_email:
        # daemon=True means this thread won't prevent Python from exiting
        # when the notebook kernel shuts down
        test_thread = threading.Thread(
            target=delayed_test_email_worker,
            args=(record, target_email, 120),   # 120 seconds = 2 minutes
            name="ContractTestEmailThread",
            daemon=True,
        )
        test_thread.start()

        print(f"      ✅ Background thread dispatched → '{test_thread.name}'")
        print(f"      📧 Test email will arrive at {target_email} in ~2 minutes")
        print(f"      ℹ️  You can run other cells — this thread works independently")
    else:
        print("      ⚠️  EMAIL_RECEIVER not set in .env — skipping test email")
        print("      Add EMAIL_RECEIVER=you@gmail.com to your .env file to enable this")

    # ── Step 5: Check for scheduled reminders due today ───────────────────────
    print("\n[5/5] Checking for scheduled reminders due today...")
    run_reminder_job(user_id=user_id)

    # ── Done — return immediately ─────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"  ✅ Pipeline complete — {Path(file_path).name}")
    print(f"  Finished : {datetime.now().strftime('%H:%M:%S')}")
    print(f"  Note     : Background email thread is still running if dispatched")
    print(f"{'=' * 60}\n")

    return record


# ── Run the pipeline ──────────────────────────────────────────────────────────
CONTRACT_DIR = Path("../data/raw/contracts")
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = list(CONTRACT_DIR.glob("*.pdf"))

if pdf_files:
    for pdf in pdf_files:
        result = full_contract_pipeline(str(pdf), user_id="notebook_test")
else:
    print(f"No PDFs found in {CONTRACT_DIR}")
    print("Running pipeline on dummy text to verify the full flow...\n")

    # ── Dummy pipeline (no real PDF needed) ───────────────────────────────────
    dummy_data   = extract_from_text(sample_text)
    dummy_record = save_contract_to_supabase(
        data=dummy_data,
        filename="dummy_contract.pdf",
        user_id="pipeline_test",
        raw_text=sample_text,
    )

    print(f"\nDummy record saved → ID: {dummy_record['id']}")

    # Manually trigger the background thread for the dummy run
    target_email = os.getenv("EMAIL_RECEIVER")
    if target_email:
        test_thread = threading.Thread(
            target=delayed_test_email_worker,
            args=(dummy_record, target_email, 120),
            name="DummyTestEmailThread",
            daemon=True,
        )
        test_thread.start()
        print(f"📧 Background thread started → test email in ~2 min to {target_email}")
    else:
        print_reminder(dummy_record, "TEST")

No PDFs found in ..\data\raw\contracts
Running pipeline on dummy text to verify the full flow...

Saved to Supabase
Contract ID : f63a2354-d950-481a-b4fc-04ff55f31bfb
Parties : Acme Technologies Pvt. Ltd. ↔ BuildRight Solutions Pvt. Ltd.
Renewal date: 2025-12-31

✅ Dummy record saved → ID: f63a2354-d950-481a-b4fc-04ff55f31bfb

[DummyTestEmailThread] 🕐 Background thread started
[DummyTestEmailThread] Will fire test email in 120 seconds (2 min)...
📧 Background thread started → test email in ~2 min to mittalishank@gmail.com


In [13]:
def view_all_contracts(user_id: str = "default") -> None:
    """Display all contracts in a clean table format."""
    import pandas as pd

    response = (
        supabase.table("contracts")
        .select("*")
        .eq("user_id", user_id)
        .order("renewal_date", desc=False)
        .execute()
    )

    if not response.data:
        print(f"No contracts found for user_id='{user_id}'")
        return

    df = pd.DataFrame(response.data)

    # Select only the readable columns for display
    display_cols = [
        "id", "filename", "party_a", "party_b",
        "contract_type", "start_date", "end_date",
        "renewal_date", "auto_renewal", "notice_period_days",
        "reminder_sent_30", "reminder_sent_7", "reminder_sent_1",
    ]
    display_cols = [c for c in display_cols if c in df.columns]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 30)
    pd.set_option("display.width", 200)

    print(f"\n📋 Contracts for user_id='{user_id}' ({len(df)} total):")
    print(df[display_cols].to_string(index=False))


# ── View all test contracts ────────────────────────────────────────────────────
view_all_contracts(user_id="notebook_test")
view_all_contracts(user_id="pipeline_test")


📋 Contracts for user_id='notebook_test' (1 total):
                                  id            filename                     party_a                        party_b     contract_type start_date   end_date renewal_date  auto_renewal  notice_period_days  reminder_sent_30  reminder_sent_7  reminder_sent_1
dfabc711-3ec1-4870-a2cd-53cf973748e1 sample_contract.pdf Acme Technologies Pvt. Ltd. BuildRight Solutions Pvt. Ltd. Service Agreement 2025-01-01 2025-12-31   2025-12-31          True                  30             False            False            False

📋 Contracts for user_id='pipeline_test' (1 total):
                                  id           filename                     party_a                        party_b     contract_type start_date   end_date renewal_date  auto_renewal  notice_period_days  reminder_sent_30  reminder_sent_7  reminder_sent_1
f63a2354-d950-481a-b4fc-04ff55f31bfb dummy_contract.pdf Acme Technologies Pvt. Ltd. BuildRight Solutions Pvt. Ltd. Service Agreement


[DummyTestEmailThread] ⏰ Delay complete — firing test email now...
✅ Email sent → mittalishank@gmail.com  (days_ahead=TEST)
[DummyTestEmailThread] ✅ Test email delivered to mittalishank@gmail.com
[DummyTestEmailThread] 🏁 Background thread finished


In [ ]:
# ── Run this cell ONLY to wipe test data from Supabase ───────────────────────
def cleanup_test_contracts(user_ids: list = ["notebook_test", "pipeline_test"]) -> None:
    for uid in user_ids:
        res = supabase.table("contracts").delete().eq("user_id", uid).execute()
        print(f"🗑️  Deleted {len(res.data)} contracts for user_id='{uid}'")

# Uncomment the line below to actually run cleanup
# cleanup_test_contracts()
print("Cleanup function defined. Uncomment the last line to run it.")